# Topic: LLM-as-a-Judge & G-Eval

## Definition (30-second explanation)
Evaluating generative AI is like judging an essay contest: there isn't one "correct" answer, and exact-word matching fails completely. 
* **LLM-as-a-Judge** is the practice of using a powerful model (like GPT-4) to grade the outputs of other models based on a rubric. 
* **G-Eval** is a specific, highly rigorous framework for this. Instead of just asking the LLM for a blind score, G-Eval forces the LLM to write down its reasoning steps (Chain-of-Thought) and then calculates a final, continuous score by looking at the mathematical *probabilities* of the score tokens (e.g., 80% sure it's a 4, 20% sure it's a 5 = 4.2).

## Why Interviewers Ask This
* Standard NLP metrics (BLEU, ROUGE) are obsolete for modern GenAI. 
* Building a reliable LLM evaluator is incredibly difficult due to inherent model biases. Interviewers want to see if you understand *Pairwise vs. Single-Answer* scoring and how to mitigate the catastrophic blind spots (position bias, verbosity bias) that ruin naive evaluation pipelines.

## Core Concepts (The 3-Layer Anatomy)
* **The Bottleneck:** Human evaluation is too slow/expensive for CI/CD. Naive LLM prompts (e.g., "Rate this 1 to 5") produce highly unstable, discrete integers resulting in massive leaderboards of ties.
* **The Mechanism (G-Eval):** 
  1. **Criteria Definition:** Provide a task (e.g., Coherence).
  2. **Chain-of-Thought (CoT):** The LLM generates step-by-step grading instructions.
  3. **Form-Filling:** The LLM evaluates the text using those steps.
  4. **Probability-Weighted Scoring:** Instead of taking the final generated string ("4"), it inspects the API's `logprobs` for tokens "1" through "5", converts them to probabilities, and calculates a weighted continuous score.
* **The Trade-off:** Relying on token probabilities requires API access to `logprobs`, which isn't supported by all model endpoints. Furthermore, LLM judges still possess inherent biases that require complex mitigation architectures.

## Pairwise Comparison vs. Single-Answer (Absolute)
* **Single-Answer (Absolute):** The judge grades Output A on a rubric (1-10). *Problem:* LLMs are bad at absolute numerical calibration.
* **Pairwise (Relative):** The judge is given Output A and Output B and asked "Which is better?" *Advantage:* LLMs are much better at relative comparisons. *Problem:* It costs 2x API calls and suffers from Position Bias.

## Critical Biases & How to Mitigate Them
* **Position Bias:** In Pairwise evaluation, the LLM will disproportionately favor whichever answer is presented first (Output A).
  * *Fix:* Run the evaluation twice (A vs. B, then B vs. A) and average the results, or randomize the order and track positions.
* **Verbosity Bias:** LLMs equate "longer" with "better" and will consistently score a verbose, rambling answer higher than a concise, accurate one.
  * *Fix:* Add strict length penalties to the rubric prompt, or normalize the final score by token count.
* **Self-Preference Bias:** GPT-4 will naturally prefer answers generated by GPT-4 over those from Claude 3, even if Claude's is objectively better.
  * *Fix:* Use an ensemble of different models as judges (e.g., GPT-4, Claude 3.5 Sonnet, Llama-3) and aggregate their scores.

## When to Use
* Evaluating open-ended generative tasks (Summarization, Translation, Creative Writing).
* Running automated regression testing in CI/CD pipelines to ensure a new prompt/model swap didn't degrade answer quality.

## Advantages
* **Human Alignment:** Highly correlated with human expert judgments compared to traditional metrics.
* **Continuous Scoring:** G-Eval's probability-weighted math turns discrete buckets (1, 2, 3) into continuous distributions (e.g., 4.35), solving the "tie" problem.

## Limitations
* **Cost:** Running a large model with CoT over thousands of logs is incredibly expensive.
* **API Dependency:** If an API provider disables `logprobs` or changes tokenization, the G-Eval mathematical scoring breaks.

## Python Syntax
```python
import numpy as np
import openai

# Minimal G-Eval Probability-Weighted Scoring Implementation
# Requires OpenAI API with logprobs enabled
client = openai.Client()

prompt = "Evaluate the coherence of this summary on a scale of 1 to 5. Summary: [TEXT]..."

# 1. Call API, requesting only 1 token (the score), but with top_logprobs enabled
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1, 
    logprobs=True, 
    top_logprobs=5 # Fetch the top 5 most likely tokens the LLM considered
)

# 2. Extract the log probabilities for the generated tokens
top_logprobs = response.choices[0].logprobs.content[0].top_logprobs
score_map = {"1": 1, "2": 2, "3": 3, "4": 4, "5": 5}

weighted_score = 0
total_prob = 0

# 3. Iterate through the top tokens, convert logprob to standard probability
for item in top_logprobs:
    if item.token in score_map:
        prob = np.exp(item.logprob) 
        weighted_score += score_map[item.token] * prob
        total_prob += prob

# 4. Calculate the weighted continuous G-Eval score
final_score = (weighted_score / total_prob) if total_prob > 0 else 0
print(f"G-Eval Continuous Score: {final_score:.2f}")
```

## 45-Second Interview Answer
"LLM-as-a-judge is the standard for evaluating open-ended GenAI text, as exact-match metrics like BLEU fail to capture semantic quality. The most rigorous framework is G-Eval, which forces the LLM to use Chain-of-Thought reasoning and calculates a continuous score using the weighted probabilities of the output tokens, rather than relying on a discrete, unstable integer. When implementing this in production, I optimize for Pairwise comparisons because LLMs are better at relative grading than absolute scoring, but I ensure we run A/B and B/A swaps to mitigate Position Bias, and apply strict prompt penalties to prevent Verbosity Bias from skewing the results."